# HybridFER-4E — Ensemble (메인, val 0.8692)

4클래스 얼굴 감정: anger / happy / panic / sadness.

구성: ResNet50 (0.041) + EfficientNet-B0 (0.073) + ViT-B/16 (0.523) + SigLIP KD (0.363). weighted soft voting + MTCNN auto crop.

- Repo: https://github.com/moneyally/yua-encoder
- Release: https://github.com/moneyally/yua-encoder/releases/tag/v1.0.0

첫 predict 호출 때 TF XLA JIT compile 로 약 100초 멈춤. 그 뒤 warm 호출은 0.7초 이하. Colab 런타임은 GPU 모드로.

## Cell 1. 환경 체크

In [ ]:
import sys, os, platform
IS_COLAB = 'google.colab' in sys.modules
print(f'python {platform.python_version()}  colab={IS_COLAB}')
if IS_COLAB:
    import subprocess
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
    print(f'gpu    {r.stdout.strip() or "GPU 없음 → Runtime > GPU 로 변경"}')
else:
    try:
        import torch
        print(f'torch  {torch.__version__}  cuda={torch.cuda.is_available()}')
    except ImportError:
        print('torch 없음. Cell 3 에서 설치.')

## Cell 2. Repo 확보

Colab 은 `/content/yua-encoder` 로 clone. 로컬은 현재 위치에서 predict.py 상위로 올라가 찾음.

In [ ]:
from pathlib import Path
if IS_COLAB:
    REPO = Path('/content/yua-encoder')
    if not REPO.is_dir():
        !git clone --depth 1 https://github.com/moneyally/yua-encoder.git {REPO}
    os.chdir(REPO)
else:
    REPO = Path.cwd()
    while not (REPO / 'predict.py').is_file() and REPO.parent != REPO:
        REPO = REPO.parent
    os.chdir(REPO)
print(f'repo   {REPO}')
assert (REPO / 'predict.py').is_file(), 'predict.py 없음'

## Cell 3. 의존성 설치 (Colab 전용)

torch 는 Colab 기본. TF / timm / transformers / facenet-pytorch 만 추가. 설치 후 'RESTART RUNTIME' 경고 뜨면 상단 메뉴 Runtime > Restart runtime 한 번 눌러서 Cell 4 부터 재실행.

In [ ]:
if IS_COLAB:
    !pip install -q \
        'tensorflow==2.21.0' \
        'timm==1.0.26' \
        'transformers==5.5.4' \
        'facenet-pytorch==2.6.0' \
        'Pillow' 'opencv-python-headless' 'scikit-learn'
    try:
        import tensorflow as tf
        print(f'install done. tf {tf.__version__}')
    except Exception as e:
        print(f'tf import 실패: {e!r}')
        print('Runtime > Restart runtime 후 Cell 4 부터 재시작.')
else:
    print('로컬 — 기존 env 사용.')

## Cell 4. Release v1.0.0 모델 다운로드

5개 파일, 약 1.03GB. Colab 에서 2~3분. 로컬엔 이미 있으면 SHA256 해시만 확인하고 넘어감.

In [ ]:
import hashlib
import urllib.request

MODELS_DIR = REPO / 'models'
MODELS_DIR.mkdir(exist_ok=True)
BASE = 'https://github.com/moneyally/yua-encoder/releases/download/v1.0.0'

sums_path = REPO / 'SHA256SUMS.txt'
if not sums_path.is_file():
    urllib.request.urlretrieve(f'{BASE}/SHA256SUMS.txt', sums_path)
expected = {}
with open(sums_path) as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith('#'):
            continue
        h, p = line.split(None, 1)
        expected[Path(p).name] = h
print(f'SHA256 사양 {len(expected)}개')

TARGETS = [
    'ensemble_with_kd.json',
    'exp02_resnet50_ft_crop_aug.h5',
    'exp04_effnet_ft_balanced.h5',
    'exp05_vit_b16_two_stage.pt',
    'exp09_siglip_kd_tsoff_T4_a07_uf4.pt',
]

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(1 << 20)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

for name in TARGETS:
    out = MODELS_DIR / name
    if out.is_file() and expected.get(name) == sha256_of(out):
        print(f'  [cached] {name}')
        continue
    print(f'  [dl]     {name}', end=' ', flush=True)
    urllib.request.urlretrieve(f'{BASE}/{name}', out)
    got = sha256_of(out)
    ok = (expected.get(name) == got)
    print('ok' if ok else f'hash mismatch ({got[:12]}...)')
    assert ok, f'{name} SHA256 불일치'
print('done')

## Cell 5. 모델 로드

여기는 weights 만 로드. XLA compile 은 Cell 6 첫 predict 에서 터짐.

In [ ]:
import time, sys
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from predict import load_model, predict, predict_probs, CLASSES

print(f'classes {CLASSES}')
t0 = time.perf_counter()
model = load_model(str(MODELS_DIR / 'ensemble_with_kd.json'),
                   tta=False, auto_face_crop=True)
print(f'load {time.perf_counter() - t0:.1f}s  type={model["type"]}')

## Cell 6. 1장 추론

순서: `IMAGE_PATH` 직접 지정 → 레포의 val 샘플 자동 픽 → Colab 업로드 위젯.

첫 호출은 XLA compile 로 100초 정도 멈춤. 아래 코드는 warmup 한 번 돌리고 두 번째 호출 시간을 찍음.

In [ ]:
IMAGE_PATH = None  # 자기 이미지 경로 (예: '/content/my_face.jpg')

if IMAGE_PATH is None:
    val_dir = REPO / 'data_rot/img/val'
    if val_dir.is_dir():
        for cls_dir in sorted(val_dir.iterdir()):
            for ip in sorted(cls_dir.glob('*.jpg'))[:1]:
                IMAGE_PATH = str(ip)
                break
            if IMAGE_PATH:
                break

if IMAGE_PATH is None and IS_COLAB:
    print('파일 올려. 얼굴 사진 1장.')
    from google.colab import files
    up = files.upload()
    if up:
        IMAGE_PATH = '/content/' + list(up.keys())[0]

if IMAGE_PATH is None:
    print('IMAGE_PATH 미지정. 맨 위 변수 직접 수정하거나 업로드 필요.')
else:
    print(f'input  {IMAGE_PATH}')
    print('warmup 1회 — 첫 호출 약 100s (XLA compile)')
    _ = predict(model, IMAGE_PATH)
    t0 = time.perf_counter()
    label, conf = predict(model, IMAGE_PATH)
    dt = time.perf_counter() - t0
    probs = predict_probs(model, IMAGE_PATH)
    print(f'pred   {label}  conf={conf:.3f}  warm={dt*1000:.0f}ms')
    for c, p in zip(CLASSES, probs):
        bar = '█' * int(p * 40)
        print(f'  {c:8s} {p:.3f}  {bar}')

## Cell 7. val 20장 정확도 (옵션)

`data_rot/img/val` 있을 때만. Colab 에서는 skip.

In [ ]:
import random
import numpy as np

val_dir = REPO / 'data_rot/img/val'
if not val_dir.is_dir():
    print('val 폴더 없음. skip.')
else:
    rng = random.Random(42)
    samples = []
    for cls in CLASSES:
        imgs = sorted((val_dir / cls).glob('*.jpg'))
        samples += [(str(p), cls) for p in rng.sample(imgs, 5)]
    y_true, y_pred = [], []
    for path, gt in samples:
        p = predict_probs(model, path)
        y_true.append(CLASSES.index(gt))
        y_pred.append(int(np.argmax(p)))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    acc = float((y_true == y_pred).mean())
    print(f'acc (n=20) {acc:.3f}')
    print('confusion matrix (row=gt, col=pred)')
    print('        ' + '  '.join(f'{c:>7s}' for c in CLASSES))
    cm = np.zeros((4, 4), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    for i, c in enumerate(CLASSES):
        print(f'{c:>7s} ' + '  '.join(f'{v:>7d}' for v in cm[i]))

---

val 1200 기준 accuracy 0.8692, Macro F1 0.8697. Warm p95 0.69s. TF 없는 환경이면 `02_vit_single_colab.ipynb` 쪽 (ViT 단일, val 0.8458).